<img src="https://raw.githubusercontent.com/PrimeReSolutions/python_actuarios/main/datos/img/logo_curso.png" width="450">

<p style="font-family: Arial; color: navy; text-align: center; font-size: 13px; letter-spacing: 1px; text-transform: uppercase; margin-bottom: 0;">
WSP — Python aplicado a modelos actuariales
</p>

<h1 style="background-color:#0070C0; color:white; text-align:center; font-family:Arial; padding:18px 0; border-radius:6px; margin-top:6px;">
🧹 Sesión 2 — Calidad de datos de reservas
</h1>

<div style="outline: 2px solid #EFB400; color:#000000; font-family: Arial; padding: 14px 18px; border-radius: 6px; margin-top: 14px;">
<h3 style="margin-top:0;">🎯 Objetivos de la sesión</h3>
<ul>
<li>Importar archivos Excel y CSV con pandas, detectando su codificación y separador.</li>
<li>Aplicar formato fecha y numérico a columnas que llegan como texto.</li>
<li>Construir validaciones de calidad de datos típicas de un cierre de reservas: vigencia, duplicados, montos, siniestralidad y coherencia de fechas.</li>
<li>Cuantificar el impacto de cada incidencia sobre el total de primas / suma asegurada y clasificarla en leve, moderado o grave.</li>
<li>Exportar un reporte de incidencias y una base depurada, usando <code>groupby</code> y máscaras booleanas — las mismas operaciones que hoy haces con tablas dinámicas y filtros en Excel.</li>
</ul>
</div>

<div style="outline: 2px solid #0070C0; color:#000000; font-family: Arial; padding: 14px 18px; border-radius: 6px; margin-top: 12px;">
<h3 style="margin-top:0;">✍️ Cómo usar este notebook</h3>
<p>Los huecos que debes completar están marcados con <code>***</code>: puede ser un nombre de columna, un operador, un número o una palabra. El código que lo rodea ya está resuelto — tu trabajo es descifrar qué falta para que la celda haga lo que dice el comentario de arriba. El notebook <b>solucionario</b> (<code>solucionarios/sesion_02_calidad_datos_solucionario.ipynb</code>) tiene la respuesta completa de cada <code>***</code>, con la misma estructura de celdas que este notebook. Ejecuta las celdas en orden: varias validaciones reutilizan un DataFrame construido en la celda anterior (<code>rrc0</code>, <code>rrc1</code>, <code>rrc2</code>, …, <code>rm0</code>, <code>rm1</code>, <code>rm2</code>, …).</p>
</div>


<h1 style="outline: 2px solid #EFB400; color: #000000; text-align: left; font-family: Arial; padding: 12px; border-radius: 6px;">
📚 Librerías
</h1>

<p style = "font-family: Arial; color:black;"> En este bloque se importan las librerías de python que utilizamos para el cálculo de reserva. Estas librerías nos permiten trabajar con diferentes estructuras de datos, realizar distintas operaciones entre ellas y graficar.
</div>

### ⚙️ Celda de arranque

Si trabajas en **Google Colab**, ejecuta esta celda **antes que cualquier otra**: descarga los datos del curso desde GitHub y deja el notebook listo para leerlos. Vuelve a ejecutarla cada vez que Colab reinicie el entorno. En tu PC (instalación local) no hace nada.

In [ ]:
# ⚙️ Celda de arranque: prepara el entorno en Google Colab (en tu PC no hace nada)
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/python_actuarios"):
        !git clone -q --depth 1 https://github.com/PrimeReSolutions/python_actuarios.git /content/python_actuarios
    %cd /content/python_actuarios/notebooks

In [ ]:
import numpy as np
import pandas as pd
import datetime
from pathlib import Path

<h1 style="outline: 2px solid #EFB400; color: #000000; text-align: left; font-family: Arial; padding: 12px; border-radius: 6px;">
📚 Parámetros
</h1>

In [ ]:
fecha_calculo = datetime.datetime.strptime('***', '%Y-%m-%d')

# Carpeta donde viven los datos del curso (sin depender de Google Drive/Colab)
DATOS = Path("../datos") if Path("../datos").exists() else Path("datos")

<h1 style="outline: 2px solid #EFB400; color: #000000; text-align: left; font-family: Arial; padding: 12px; border-radius: 6px;">
🧭 De Excel a pandas
</h1>

Todo lo que hoy haces en Excel para depurar una base de pólizas tiene un equivalente directo en pandas — esta tabla es tu diccionario de traducción. Vuelve a ella cada vez que te preguntes "¿esto cómo se hacía en Excel?". No hace falta memorizarla: cada fórmula aparece en su momento, con un ejemplo resuelto igual que en un archivo de reservas real.

| En Excel | En pandas | Dónde lo vas a usar hoy |
|---|---|---|
| `BUSCARV` / `XLOOKUP` | `df['col'].isin(...)` / `pd.merge()` | Cruzar pólizas contra la base de siniestros |
| Tabla dinámica | `groupby().agg()` | Resumen de incidencias por producto y tipo de error |
| `SUMAR.SI.CONJUNTO` / `CONTAR.SI.CONJUNTO` | `groupby().agg({'col':'sum'/'count'})` | Monto y número de pólizas por tipo de incidencia |
| Filtros / Autofiltro | Máscaras booleanas `df[condición]` | Cada validación de calidad de datos de esta sesión |
| `Ctrl+T` (convertir en tabla) | `DataFrame` | La estructura que usamos de principio a fin |


---
# **1. IMPORTACIÓN DE DATOS**
---

## 📂 Importar un archivo XLSX

∫🐍 **Fórmula**

```python
df = pd.read_excel('ruta\\NombreArchivo.xlsx', header=None, index_col=0, dtype=None, sheet_name='Hoja1')
```

<code>header</code>: <span style="font-size:7px;">Nombre de las columnas. Default: primera fila</span>

<code>index_col</code> <span style="font-size:7px;">Nombre de las filas. Default: [0, n]</span>

<code>dtype</code> <span style="font-size:7px;">Tipo de datos de las columnas. Default: None. *Object conserva los datos tal y como están Excel, sin interpretación.* </span>

<code>sheet_name</code> <span style="font-size:7px;">Nombre de la hoja donde se encuentran los datos</span>

1️⃣ Ruta del archivo

In [ ]:
ruta_rrc = DATOS / "base_rrc.xlsx"

2️⃣ Importar el archivo

In [ ]:
base_rrc_ori = pd.read_excel(ruta_rrc, dtype={'monto prima': 'object', 'gastos_adq':'object'})

🔎 Visualizar datos

In [ ]:
base_rrc_ori.head()

## 📂 Importar un archivo CSV



∫🐍 **Fórmula**

```python
df = pd.read_csv( 'ruta \\ NombreArchivo.csv', sep = ",", encoding = "latin", header = None, index_col = 0, dtype=None)
```

<code>sep</code>: <span style="font-size:7px;"> Separador de columnas</span>

<code>encoding</code> <span style="font-size:7px;"> Formato del archivo: 'latin-1', 'UTF-08' </span>

<code>header</code> <span style="font-size:7px;"> Nombre de las columnas. Default: primera fila </span>

<code>index_col</code> <span style="font-size:7px;"> Nombre de las filas. Default: [0, n] </span>

<code>dtype</code> <span style="font-size:7px;">Tipo de datos de las columnas. Default: None. *Object conserva los datos tal y como están Excel, sin interpretación.* </span>

1️⃣ Ruta del archivo

In [ ]:
ruta_rmat = DATOS / "base_rmat.csv"

2️⃣ Verificar que el archivo existe localmente

In [ ]:
archivo_local = ruta_rmat

if archivo_local.exists():
    print("✅ Archivo disponible en:", archivo_local)
else:
    raise FileNotFoundError(f"No se encontró el archivo {archivo_local}")

3️⃣ Identificar la codificación de caracteres del archivo csv

In [ ]:
import chardet

with open(archivo_local, 'rb') as f:
    rawdata = f.read(20000)
result = chardet.detect(rawdata)
encoding_detected = result['encoding']

print("✅ Codificación detectada:", encoding_detected)

4️⃣ Identificar el separador de columnas

In [ ]:
for sep in [',', ';', '|', '\t']:
    try:
        df = pd.read_csv(archivo_local, sep=sep, encoding = encoding_detected)
        print(f"✅ Separador '{sep}' funciona, columnas: {df.columns[:].tolist()}")
        sep_detected = sep
        break
    except Exception as e:
        continue

5️⃣ Importar archivo

In [ ]:
base_rmat_ori = pd.read_csv(archivo_local, sep=sep_detected, encoding=encoding_detected, dtype={'gastos_adm': 'object', 'monto prima': 'object', 'suma_asegurada':'object'} )

🔎 Visualizar datos

In [ ]:
base_rmat_ori.head()

---
# **2. TRATAMIENTO DE LAS BASES DE DATOS**
---

## 📄 Archivo de rrc

In [ ]:
base_rrc = base_rrc_ori.copy()
base_rrc.head()

🔍🧾 Revisión del formato del archivo

In [ ]:
base_rrc.dtypes

#### *Formato fecha 📅*

📅 **Formato de fecha esperado — `base_rrc.xlsx`**

Al leer el Excel, las columnas `fecha inicio` y `fecha fin` ya llegan como fecha nativa de Excel (`datetime64`). Si se leyeran como texto, el formato sería ISO `%Y-%m-%d` (ej. `2024-05-26`).

In [ ]:
base_rrc['fecha inicio'] = pd.to_datetime(base_rrc['fecha inicio'], format = "***", errors="coerce")
base_rrc['fecha fin'] = pd.to_datetime(base_rrc['fecha fin'], format = "***", errors="coerce")
base_rrc.head()

#### *Formato numérico 🔢*

In [ ]:
base_rrc['monto prima'] = pd.to_numeric(base_rrc['***'])
base_rrc['gastos_adq'] = base_rrc['gastos_adq'].***(float)
base_rrc.head()

🔍🧾 Revisión del nuevo formato del archivo

In [ ]:
base_rrc.dtypes

---
## 📄 Archivo de rmat

In [ ]:
base_rmat = base_rmat_ori.copy()
base_rmat.head()

🔍🧾 Revisión del formato del archivo

In [ ]:
base_rmat.dtypes

#### *Formato fecha 📅*

📅 **Formato de fecha esperado — `base_rmat.csv`**

Las columnas `fecha nacimiento`, `fecha inicio` y `fecha fin` vienen como texto en formato ISO `%Y-%m-%d` (ej. `1951-03-03`). *Nota:* este CSV es un **sustituto** de la base original (ver `datos/README.md`); a diferencia de la base original ya no usa formato `%m/%d/%Y`.

In [ ]:
base_rmat['fecha nacimiento'] = pd.to_datetime(base_rmat['fecha nacimiento'], format = "***", errors="coerce")
base_rmat['fecha inicio'] = pd.to_datetime(base_rmat['fecha inicio'], format = "***", errors="coerce")
base_rmat['fecha fin'] = pd.to_datetime(base_rmat['fecha fin'], format = "***", errors="coerce")

base_rmat.head()

#### *Formato numérico 🔢*

In [ ]:
base_rmat['gastos_adm'] = base_rmat['gastos_adm'].***(float)
base_rmat['monto prima'] = base_rmat['monto prima'].***(float)

colSA = base_rmat.columns[base_rmat.columns.str.contains(pat = '***')]   # todas las columnas de suma_asegurada
base_rmat[colSA] = base_rmat[colSA].apply(pd.to_numeric, errors='coerce')

🔍🧾 Revisión del nuevo formato del archivo

In [ ]:
base_rmat.dtypes

---
# **3. CALIDAD DE DATOS**
---

## 📄 ARCHIVO DE RESERVA DE RIESGOS EN CURSO (RRC)

##### ⚠️ 1. Identificar pólizas no vigentes

Una póliza es "vigente" a la fecha de cálculo cuando `fecha inicio <= fecha_calculo < fecha fin`. Si constituyes reserva de riesgos en curso (RRC) para una póliza que ya venció, o para una que todavía no ha empezado, estás reservando por un riesgo que la aseguradora no está corriendo — infla el pasivo sin motivo. Este es, casi siempre, el primer filtro de cualquier cierre: antes de calcular montos hay que asegurarse de que el universo de pólizas es el correcto.


In [ ]:
#Creamos una copia del archivo con el formato validado y listo para trabajar
rrc0 = base_rrc.copy()

#Aplicamos los criterios de fecha: Fecha inicio <= Fecha de calculo < Fecha fin
rrc0.loc[rrc0['fecha inicio'] < rrc0['fecha fin'], 'F.inicio < F.fin'] = 'cumple'
rrc0.loc[rrc0['fecha inicio'] >= rrc0['fecha fin'], 'F.inicio < F.fin'] = 'no cumple'

rrc0.loc[rrc0['fecha inicio'] <= fecha_calculo, 'F.inicio < F.cal'] = 'cumple'
rrc0.loc[rrc0['fecha inicio'] > fecha_calculo, 'F.inicio < F.cal'] = 'no cumple'

rrc0.loc[fecha_calculo < rrc0['fecha fin'], 'F.cal < F.fin'] = 'cumple'
rrc0.loc[fecha_calculo >= rrc0['fecha fin'], 'F.cal < F.fin'] = 'no cumple'

#Filtramos las pólizas vigentes: fecha inicio <= fecha_calculo Y fecha_calculo < fecha fin
rrc1 = rrc0[(rrc0['***'] <= fecha_calculo) & (fecha_calculo < rrc0['***'])]               # base vigente

In [ ]:
#Identificamos las pólizas no vigentes y las guardamos en el df "error1"
error1 = rrc0[(rrc0['fecha inicio'] > fecha_calculo) | (fecha_calculo >= rrc0['fecha fin'])].copy()      # base con errores
error1['tipo'] = '***'
error1

##### ⚠️ 2. Identificar pólizas duplicadas

Si la misma póliza aparece dos veces en la base — por un reproceso de sistemas o una carga duplicada — su reserva se cuenta dos veces en el balance: una doble contabilización que sobreestima el pasivo y, si nadie lo detecta, se arrastra cierre tras cierre. Usamos `nombre_producto + num_poliza` como llave porque el mismo número de póliza puede repetirse legítimamente entre productos distintos.


In [ ]:
#Creamos una copia de la base de pólizas vigentes
rrc2 = rrc1.copy()

#Filtramos las pólizas sin duplicado considerando la llave "nombre_producto" y "num_poliza"
rrc2['llave'] = rrc2['***'].astype(str) + rrc2['num_poliza'].astype(str)
rrc3 = rrc2[rrc2['llave'].duplicated() == False].reset_index(drop=True)

In [ ]:
#Identificamos las pólizas duplicadas y las guardamos en el df "error2"
error2 = rrc2[rrc2['llave'].duplicated() == ***].reset_index(drop=True)
error2['tipo'] = 'duplicado'
error2

##### ⚠️ 3.1 Identificar montos vacíos

Una prima vacía (`NaN`) casi siempre es un problema de origen: un registro que no migró completo desde el sistema administrador. No se puede calcular una reserva sobre un monto que no existe, así que estas pólizas se separan para investigarlas — sumarlas como cero subestimaría la reserva, e ignorarlas sin dejar registro es peor.


In [ ]:
#Creamos una copia de la base de pólizas vigentes y sin duplicados
rrc4 = rrc3.copy()

#Filtramos las pólizas que no tienen valores vacíos en la columna "monto prima"
rrc5 = rrc4[~rrc4['***'].isnull()]

In [ ]:
#Identificamos las pólizas con vacíos y las guardamos en el df "error3_1"
error3_1 = rrc4[rrc4['monto prima'].isnull()] .copy()
error3_1['tipo'] = '***'
error3_1

##### ⚠️ 3.2 Identificar montos inconsistentes

Una prima negativa o cero no representa un riesgo asegurado real; casi siempre es una reversión, un ajuste contable mal aplicado o un error de captura. Dejarla dentro del cálculo de RRC puede compensar artificialmente otras pólizas y esconder el tamaño real de la reserva.


In [ ]:
#Creamos una copia de la base de pólizas vigentes, sin duplicados y sin montos vacíos en la columna "prima"
rrc6 = rrc5.copy()

#Filtramos las pólizas que tienen valores positivos en la columna "monto prima"
rrc = rrc6[rrc6['monto prima'] > ***]

In [ ]:
#Identificamos las pólizas con montos negativos y las guardamos en el df "error3_2"
error3_2 = rrc6[rrc6['monto prima'] <= ***].copy()
error3_2['tipo'] = 'monto negativo' 

In [ ]:
#Agrupamos los errores de montos inconsistentes
error3 = pd.concat([***, ***], axis=0)
error3

##### ⚠️ 4. Identificar pólizas siniestradas

Una póliza con un siniestro ya reportado dejó de correr el riesgo por el que se constituye la RRC: ese riesgo ya se materializó y ahora se refleja en la reserva de siniestros, no en la de riesgos en curso. Si no cruzas contra la base de siniestros, terminas reservando dos veces el mismo riesgo — una vez como RRC y otra como reserva de siniestros.


In [ ]:
#Importamos el archivo de siniestros
ruta_siniestros = DATOS / "siniestros.xlsx"
base_siniestros = pd.read_excel(ruta_siniestros)

In [ ]:
#Creamos el df "siniestrados" con los num_poliza de las pólizas de la base de siniestros
siniestrados = base_siniestros['num_poliza']

#Identificamos las pólizas de la base de siniestros y las guardamos en el df "rrc_siniestrados"
rrc_siniestrados = rrc[rrc['num_poliza'].isin(siniestrados)].copy()
rrc_siniestrados['tipo'] = '***'

#Filtramos las pólizas que no aparecen en la base de siniestros
rrc = rrc[~rrc['***'].isin(siniestrados)]

#Generamos una tabla con el número de pólizas y monto de primas según nombre_producto
siniestros_rrc = rrc_siniestrados.groupby(['nombre_producto']).agg({'num_poliza':'count','monto prima':'sum'})
siniestros_rrc.loc['total'] = siniestros_rrc.sum(axis=0)

#Visualizamos con 2 decimales y separador de miles ","
pd.options.display.float_format = '{:,.2f}'.format
rrc_siniestrados

In [ ]:
base_limpia_rrc = rrc.iloc[:, :base_rrc_ori.shape[1]]
base_limpia_rrc

#### **📊 Revisión de errores**

No todas las incidencias pesan igual: 3 pólizas duplicadas sobre una cartera de 140,000 no cambian el balance, pero si esas 3 concentran el 25% de la prima, sí. Por eso ponderamos cada tipo de error por su participación en el total de primas (suma asegurada en RM) y lo clasificamos según estos umbrales:

| Impacto (% del total) | Clasificación | Lectura |
|---|---|---|
| ≤ 5% | 🟢 Leve | Se documenta y se corrige en el siguiente ciclo de datos |
| 5% – 20% | 🟡 Moderado | Se corrige antes de cerrar; puede requerir ajuste manual |
| > 20% | 🔴 Grave | Bloquea el cierre hasta resolver la causa raíz |

Son umbrales de gestión, no una norma — pero es exactamente el tipo de regla que se implementa en el motor de reservas de una aseguradora para decidir qué incidencias se pueden dejar pasar y cuáles no.


In [ ]:
#Agrupamos los 4 grupos de pólizas depuradas
error = pd.concat([error1, error2, error3, rrc_siniestrados], axis=0, ignore_index = True)

#Creamos una tabla con el número de pólizas y monto de primas según nombre_producto
resumen_incidencias = error.groupby(['***', 'tipo']).agg({'num_poliza':'count', 'monto prima':'sum'})
resumen_incidencias = resumen_incidencias.reset_index(1)

total = rrc['monto prima'].sum(axis=0) #suma de primas del archivo rrc

#Ponderamos las incidencias según el monto de primas
resumen_incidencias['impacto (%)'] = (resumen_incidencias['monto prima'] / total) * 100
resumen_incidencias.loc[resumen_incidencias['impacto (%)'] <= ***, 'impacto'] = 'leve'
resumen_incidencias.loc[(resumen_incidencias['impacto (%)'] > 5) & (resumen_incidencias['impacto (%)'] <= 20), 'impacto'] = 'moderado'
resumen_incidencias.loc[resumen_incidencias['impacto (%)'] > ***, 'impacto'] = 'grave'

resumen_incidencias.columns = [ 'Error', 'Certificados', 'Monto prima', 'impacto (%)', 'impacto']
resumen_incidencias.loc['total'] = resumen_incidencias.select_dtypes(include='number').sum()

pd.options.display.float_format = '{:,.2f}'.format
resumen_incidencias

---
## 📄 ARCHIVO DE RESERVAS MATEMÁTICAS (RM)

##### **⚠️ F1 Identificar fechas vacias**

En reservas matemáticas trabajamos con tres fechas por asegurado (nacimiento, inicio, fin) porque de ellas depende directamente la edad actuarial y la duración de la póliza: los dos insumos de cualquier tabla de mortalidad. Una fecha vacía no se puede sustituir por un supuesto — sin ella no hay forma de calcular la reserva de esa póliza, así que se separa para investigación antes de seguir.


In [ ]:
#Otras formas de filtrar un df:
#base_rmat[ (base_rmat['nombre_producto']=='producto03') & (base_rmat['sexo']=='M')]
#base_rmat.query(" nombre_producto=='producto03' and sexo=='M' ")
#base_rmat.loc[ (base_rmat['nombre_producto']=='producto03') & (base_rmat['sexo']=='M'), ['num_poliza', 'nombre_producto', 'riesgo', 'sexo', 'moneda'] ]

In [ ]:
#Creamos una copia del archivo con el formato validado y listo para trabajar
rm0 = base_rmat.copy()

#Filtramos las pólizas que no tienen valores vacíos en las columnas "fecha nacimiento", "fecha inicio" y "fecha fin"
rm1 = rm0[~( (rm0['***'].isnull()) | (rm0['fecha inicio'].isnull()) | (rm0['fecha fin'].isnull() ) )]

In [ ]:
#Identificamos las pólizas vacías y las guardamos en el df "error0_1"
error0_1 = rm0[(rm0['fecha nacimiento'].isnull()) | (rm0['fecha inicio'].isnull()) | (rm0['fecha fin'].isnull())].copy()
error0_1['tipo'] = '***'
error0_1

##### **⚠️ F2 Identificar fechas menores a 1910**

Una fecha como `1900-01-01` casi nunca es real: suele ser una fecha "centinela" que usan los sistemas antiguos cuando el campo no se llenó, o un error de digitación (un `19` que debió ser `20`). Si se cuela en el cálculo produce una edad absurda — de 100+ años — que distorsiona la mortalidad esperada de toda la cartera. 1910 es un límite razonable: nadie asegurable hoy nació antes de esa fecha.


In [ ]:
#Definimos una fecha mínima como referencia para asegurar la coherencia temporal de los datos
fecha_minima =  datetime.datetime.strptime('***', '%Y-%m-%d')

#Creamos una copia de la base de pólizas que no tienen fechas vacías
rm2 = rm1.copy()

#Filtramos las pólizas que tienen fechas coherentes
rm3 = rm2[(rm2['fecha nacimiento'] > fecha_minima) & (rm2['fecha inicio'] > fecha_minima) & (rm2['fecha fin'] > fecha_minima)]

In [ ]:
#Identificamos las pólizas con fechas inconsistentes y las guardamos en el df "error0_2"
error0_2 = rm2[(rm2['fecha nacimiento'] <= fecha_minima) | (rm2['fecha inicio'] <= fecha_minima) | (rm2['fecha fin'] <= fecha_minima)].copy()
error0_2['tipo'] = '***'
error0_2

##### **⚠️ F3 Verificar fecha nacimiento < fecha de inicio**

Es una regla de sentido común que vale la pena validar explícitamente: nadie puede contratar una póliza antes de haber nacido. Cuando esto falla, casi siempre es porque las columnas de fecha se cruzaron en la captura o en la migración — y si no se detecta, el cálculo de edad (y con él la prima de riesgo) queda invertido.


In [ ]:
#Creamos una copia de la base de pólizas que no tienen fechas vacías ni inconsistentes
rm4 = rm3.copy()

#Filtramos las pólizas que cumplen: fecha nacimiento < fecha de inicio
rm5 = rm4[(rm4['fecha nacimiento'] < rm4['***'])]

In [ ]:
#Identificamos las pólizas que no cumplen el criterio y las guardamos en el df "error0_3"
error0_3 = rm4[(rm4['fecha nacimiento'] >= rm4['fecha inicio'])]
error0_3['tipo'] = '***'
error0_3

##### **⚠️ F4 Verificar fecha nacimiento (edad) > 18**

Un asegurado menor de edad en una póliza de vida individual suele ser un error de digitación en la fecha de nacimiento, o un producto mal clasificado (por ejemplo, un seguro infantil capturado como individual). Si se queda así, distorsiona la mortalidad esperada de ese segmento de edad: la tabla `qx` que usarás más adelante en el curso asume una población adulta.


In [ ]:
#Definimos la edad mínima para estar asegurado
limite_edad = fecha_calculo - datetime.timedelta(days=365 * ***)

#Creamos una copia de la base de pólizas que no tienen fechas vacías ni inconsistentes
rm6 = rm5.copy()

#Filtramos las pólizas que tienen asegurados mayores de 18 años
rm7 = rm6[(rm6['fecha nacimiento'] <= limite_edad)]

In [ ]:
#Identificamos las pólizas con asegurados menores de edad y las guardamos en el df "error0_4"
error0_4 = rm6[(rm6['fecha nacimiento'] > limite_edad)].copy()
error0_4['tipo'] = '***' 

In [ ]:
#Agrupamos los 4 grupos de pólizas depuradas según fechas
error0 = pd.concat([***, error0_2, error0_3, ***], axis=0)
error0

##### **⚠️ 1. Identificar pólizas no vigentes**

Igual que en RRC, una póliza fuera de vigencia no debería seguir generando reserva matemática — pero aquí el efecto es mayor: la reserva matemática de vida se proyecta a varias décadas, así que dejar dentro una póliza ya vencida arrastra ese error durante todo el horizonte de cálculo.


In [ ]:
#Creamos una copia de la base de pólizas que no tienen fechas vacías ni inconsistentes
rm7 = rm7.copy()

#Aplicamos los criterios de fecha: Fecha inicio <= Fecha de calculo < Fecha fin
rm7.loc[rm7['fecha inicio'] < rm7['fecha fin'], 'F.inicio < F.fin'] = 'cumple'
rm7.loc[rm7['fecha inicio'] >= rm7['fecha fin'], 'F.inicio < F.fin'] = 'no cumple'

rm7.loc[rm7['fecha inicio'] <= fecha_calculo, 'F.inicio <= F.cal'] = 'cumple'
rm7.loc[rm7['fecha inicio'] > fecha_calculo, 'F.inicio <= F.cal'] = 'no cumple'

rm7.loc[fecha_calculo < rm7['fecha fin'], 'F.cal < F.fin'] = 'cumple'
rm7.loc[fecha_calculo >= rm7['fecha fin'], 'F.cal < F.fin'] = 'no cumple'

#Filtramos las pólizas vigentes
rm8 = rm7[(rm7['***'] <= fecha_calculo) & (fecha_calculo < rm7['***'])]

In [ ]:
#Identificamos las pólizas no vigentes y las guardamos en el df "error1"
error1 = rm7[(rm7['fecha inicio'] > fecha_calculo) | (fecha_calculo >= rm7['fecha fin'])].copy()
error1['tipo'] = '***'
error1

##### **⚠️ 2. Identificar pólizas duplicadas**

El mismo riesgo de doble contabilización de RRC aplica aquí, con un agravante: la reserva matemática crece con el tiempo (a diferencia de la RRC, que se libera conforme pasa la vigencia). Una póliza duplicada que no se corrige hoy sigue duplicada — y creciendo — en cada cierre futuro.


In [ ]:
#Creamos una copia de la base de pólizas que no tienen fechas vacías ni inconsistentes y están vigentes
rm9 = rm8.copy()

#Filtramos las pólizas sin duplicado considerando la llave "nombre_producto" y "num_poliza"
rm9['llave'] = (rm9['***'].astype(str) + rm9['num_poliza'].astype(str))
rm9_dedup = rm9[rm9['llave'].duplicated() == False].reset_index(drop=True)

In [ ]:
#Identificamos las pólizas duplicadas y las guardamos en el df "error2"
error2 = rm9[rm9['llave'].duplicated() == ***].reset_index(drop=True)
error2['tipo'] = 'duplicado'
error2

In [ ]:
#Ejemplo de duplicated(keep=)
#df = pd.DataFrame({'llave': [1, 2, 2, 3, 3, 3, 4], 'valor': ['A','B','C','D','E','F','G']})
#df

In [ ]:
#df['duplicado_first'] = df['llave'].duplicated(keep='***')
#df['duplicado_last'] = df['llave'].duplicated(keep='***')
#df['duplicado_all'] = df['llave'].duplicated(keep=***)
#df

##### **⚠️ 3.1 Identificar montos vacíos**

Aquí validamos tanto la prima como la suma asegurada: son los dos montos que alimentan el cálculo de reserva matemática. Una suma asegurada vacía es particularmente delicada porque además define el capital que la aseguradora pagaría en caso de siniestro — no hay forma de estimarla "a ojo".


In [ ]:
#Creamos una copia de la base de pólizas con fechas consistentes, vigentes y sin duplicados
rm10 = rm9_dedup.copy()

#Filtramos las pólizas que no tienen valores vacíos en las columnas "monto prima" y "suma_asegurada"
rm11 = rm10[~ ( (rm10['***'].isnull()) | (rm10['suma_asegurada'].isnull()) ) ]

In [ ]:
#Identificamos las pólizas con vacíos y las guardamos en el df "error3_1"
error3_1 = rm10[(rm10['monto prima'].isnull()) | (rm10['suma_asegurada'].isnull())].copy()  # base con errores
error3_1['tipo'] = '***'
error3_1

##### **⚠️ 3.2 Identificar montos inconsistentes**

Una prima o suma asegurada negativa o cero no corresponde a ningún producto de vida real. Igual que en RRC, dejarlas dentro del cálculo distorsiona el resultado agregado — y en RM el efecto se compone, porque la reserva se proyecta a futuro sobre esa misma base.


In [ ]:
#Creamos una copia de la base de pólizas con fechas consistentes, vigentes, sin duplicados y sin vacios
rm12 = rm11.copy()

#Filtramos las pólizas que tienen valores positivos en las columnas "monto prima" y "suma_asegurada"
rm13 = rm12[(rm12['monto prima'] > ***) & (rm12['suma_asegurada'] > ***)]

In [ ]:
#Identificamos las pólizas con montos negativos y las guardamos en el df "error3_2"
error3_2 = rm12[(rm12['monto prima'] <= ***) | (rm12['suma_asegurada'] <= ***)].copy()
error3_2['tipo'] = 'monto negativo' 

In [ ]:
#Agrupamos los errores de montos inconsistentes
error3 = pd.concat([***, ***], axis=0)
error3

##### **⚠️ Identificar pólizas siniestradas**

Una póliza de vida con siniestro pagado (fallecimiento, invalidez u otra cobertura) ya cumplió el riesgo que la reserva matemática venía provisionando: ese capital pasa a la reserva de siniestros, y mantenerlo también en RM duplicaría la obligación registrada en el balance.


In [ ]:
#Filtramos las pólizas que no aparecen en la base de siniestros
rmat = rm13[~rm13['num_poliza'].isin(siniestrados)]

#Identificamos las pólizas de la base de siniestros y las guardamos en el df "rmat_siniestrados"
rmat_siniestrados = rm13[rm13['num_poliza'].isin(siniestrados)].copy()
rmat_siniestrados['tipo'] = '***'

#Generamos una tabla con el número de pólizas y monto de primas según nombre_producto
siniestros_rmat = rmat_siniestrados.groupby(['***']).agg({'num_poliza':'count','monto prima':'sum'})
siniestros_rmat.loc['total'] = siniestros_rmat.sum(axis=0)

#Visualizamos con 2 decimales y separador de miles ","
pd.options.display.float_format = '{:,.2f}'.format
siniestros_rmat

#### **📊 Revisión de errores**

Aplicamos el mismo criterio de impacto que en RRC (los umbrales de la tabla anterior: ≤5% leve, 5–20% moderado, >20% grave), pero aquí ponderamos por **suma asegurada** en vez de por prima: en reservas matemáticas de vida es la variable que mejor refleja el compromiso real de la aseguradora con cada asegurado.


In [ ]:
#Agrupamos los 4 grupos de pólizas depuradas
error = pd.concat([error0, error1, error2, error3, rmat_siniestrados], axis=0, ignore_index = True)

#Creamos una tabla con el número de pólizas y monto de primas según nombre_producto
resumen_incidencias = error.groupby(['***', 'tipo']).agg({'num_poliza':'count', 'suma_asegurada':'sum'})
resumen_incidencias = resumen_incidencias.reset_index(1)

total = rmat['suma_asegurada'].sum(axis=0)  #suma de SA del archivo rmat

#Ponderamos las incidencias según el monto de SA
resumen_incidencias['impacto (%)'] = (resumen_incidencias['suma_asegurada'] / total) * 100

resumen_incidencias.loc[resumen_incidencias['impacto (%)'] <= ***, 'impacto'] = 'leve'
resumen_incidencias.loc[(resumen_incidencias['impacto (%)'] > 5) & (resumen_incidencias['impacto (%)'] <= 20), 'impacto'] = 'moderado'
resumen_incidencias.loc[resumen_incidencias['impacto (%)'] > ***, 'impacto'] = 'grave'

resumen_incidencias.columns = [ 'Error', 'Certificados', 'Suma asegurada', 'impacto (%)', 'impacto']
resumen_incidencias.loc['total'] = resumen_incidencias.select_dtypes(include='number').sum()

pd.options.display.float_format = '{:,.2f}'.format
resumen_incidencias

---
# **4. EXPORTACIÓN DE REPORTES**
---

1️⃣Seleccionar dimensiones del archivo para exportar

In [ ]:
#Seleccionamos únicamente las columnas originales del archivo para generar la versión final depurada
base_limpia = rmat.iloc[:, :base_rmat_ori.shape[1]].reset_index(drop=True)

2️⃣Exportación de la base depurada y hallazgos encontrados

In [ ]:
#Definir ruta de exportación (carpeta local, sin depender de Google Drive/Colab)
Path("salidas").mkdir(exist_ok=True)
ruta_exportacion = Path("salidas") / "01_reporte_de_incidencias_rmat.xlsx"

with pd.ExcelWriter(ruta_exportacion, engine='openpyxl') as writer:
    base_limpia.to_excel(writer, sheet_name="base_limpia", index=False)
    error0.to_excel(writer, sheet_name="fechas_inconsistentes", index=False)
    error1.to_excel(writer, sheet_name="no_vigentes", index=False)
    error2.to_excel(writer, sheet_name="duplicados", index=False)
    error3.to_excel(writer, sheet_name="montos_inconsistentes", index=False)
    rmat_siniestrados.to_excel(writer, sheet_name="siniestrados", index=False)

print("✅ Archivo exportado correctamente:", ruta_exportacion)

3️⃣Guardar los df para poder usarlos en otro Notebook (carpeta local `salidas/`)

In [ ]:
# En el entorno local no usamos Google Drive: nos aseguramos de que exista la carpeta salidas/
Path("salidas").mkdir(exist_ok=True)

In [ ]:
list(Path("salidas").glob("*"))

In [ ]:
import pickle

def store(var, nombre):
    with open(f'salidas/{nombre}.pkl', 'wb') as f:
        pickle.dump(var, f)

def restore(nombre):
    with open(f'salidas/{nombre}.pkl', 'rb') as f:
        return pickle.load(f)

In [ ]:
store(base_limpia, 'base_limpia_rmat')
store(base_limpia_rrc, 'base_limpia_rrc')

<h1 style="outline: 2px solid #EFB400; color: #000000; text-align: left; font-family: Arial; padding: 12px; border-radius: 6px;">
📌 Recap
</h1>

- Importaste dos bases con formatos distintos (Excel y CSV) y aprendiste a detectar codificación y separador antes de leer el archivo.
- Convirtiste texto a fecha y a número — el paso que hace posible cualquier validación posterior; sin formato correcto, una comparación de fechas o un filtro de montos simplemente no funciona.
- Construiste 5 validaciones sobre RRC y 9 sobre RM: vigencia, duplicados, montos vacíos/inconsistentes, siniestralidad y, solo en RM, coherencia de fechas de nacimiento.
- Cuantificaste el impacto de cada incidencia sobre el total de primas / suma asegurada y lo clasificaste en leve, moderado o grave — el mismo criterio que se usa para decidir si una incidencia bloquea un cierre real.
- Exportaste una base depurada y un reporte de incidencias a `salidas/`, y guardaste `base_limpia_rmat` / `base_limpia_rrc` en `salidas/*.pkl` como respaldo de tu propio trabajo — cada sesión del curso sigue cargando sus datos de forma independiente (ver convención del material).

**Siguiente parada:** Sesión 3 — Triángulos e IBNR, donde vas a aplicar la misma lógica de "detectar, cuantificar y tratar incidencias", ahora sobre triángulos de siniestros con `chainladder`.
